In [1]:
import json
from utils.gait_parameters_extractor_v2 import GaitParametersExtractorV2
from utils.gait_parameters_extractor import CoordinatesIdx

bvh_dataset = "./datasets/mocap/dataset_v3.json"

with open(bvh_dataset, 'r') as file:
    data_3d = json.load(file)

data_3d['p1s1'][0]

{'lfemur': [10.120033264160156, 3.5556223392486572, 0.022530317306518555],
 'ltibia': [10.405301094055176, 1.7651171684265137, -0.06053118407726288],
 'lfoot': [10.649447441101074, 0.23312509059906006, -0.13180023431777954],
 'ltoes': [10.166183471679688, 0.07413950562477112, -0.14978471398353577],
 'rfemur': [10.118398666381836, 3.534363269805908, -1.0120400190353394],
 'rtibia': [10.269598007202148, 1.6986266374588013, -0.8861502408981323],
 'rfoot': [10.4542236328125, 0.13499772548675537, -0.7972148656845093],
 'rtoes': [9.96440601348877, 0.12542156875133514, -0.7831996083259583],
 'lhumerus': [10.17995834350586, 5.267821788787842, 0.14029821753501892],
 'lradius': [10.326631546020508, 4.080488204956055, 0.14565566182136536],
 'lwrist': [10.24118709564209, 3.4502627849578857, 0.1927906572818756],
 'rhumerus': [10.1235933303833, 5.247973918914795, -0.9754229784011841],
 'rradius': [10.334478378295898, 3.9820308685302734, -1.089633584022522],
 'rwrist': [10.16678237915039, 3.386455059

In [2]:
from scripts.parsers import parse_sequences as parse_sequence_info

file_path = 'gait3d\\ListOfSequences.txt'
sequences = parse_sequence_info(file_path)

In [3]:
def _calc_step_frames(steps: list) -> list[int]:
    step_frames = []
    for i in range(len(steps)-1):
        step_frames.append(steps[i+1] - steps[i])
    return step_frames

def _fragment_step_frames(steps: list, window_size: int = 32) -> list[tuple[int, int]]:
    step_frames = []
    for i in range(len(steps)-1):
        center = steps[i] + (steps[i+1] - steps[i])//2
        start_frame = center - window_size//2
        end_frame = center + window_size//2
        step_frames.append((start_frame, end_frame))
    return step_frames

In [4]:
results = {}
cum_step_frames = []
side = ''

for seq_key in list(data_3d.keys()):
    gpe = GaitParametersExtractorV2(data_3d[seq_key], CoordinatesIdx(2, 0, 1), scale_factor=255)
    if gpe.l_steps[0] < gpe.r_steps[0]:
        cum_step_frames += _calc_step_frames(gpe.l_steps)
        steps_sequence = _fragment_step_frames(gpe.l_steps)
        side = 'L'
    else:
        cum_step_frames += _calc_step_frames(gpe.r_steps)
        steps_sequence = _fragment_step_frames(gpe.r_steps)
        side = 'R'

    print(f"{seq_key} [{side}] - {steps_sequence}")
    for i, (start_frame, end_frame) in enumerate(steps_sequence):
        results[f"{seq_key}c{i}"] = gpe.seq_params[start_frame:end_frame]


print("   Steps: ", len(cum_step_frames))
print("    Mean: ", sum(cum_step_frames)/len(cum_step_frames))
print("     Max: ", max(cum_step_frames))
print("     Min: ", min(cum_step_frames))
print("  Median: ", sorted(cum_step_frames)[len(cum_step_frames)//2])
print(" Over 32: ", sum( 1 for step in cum_step_frames if step>32))


p1s1 [L] - [(19, 51), (51, 83), (82, 114)]
p1s2 [L] - [(21, 53), (52, 84)]
p1s3 [L] - [(16, 48), (46, 78), (76, 108)]
p1s4 [L] - [(17, 49), (47, 79)]
p2s1 [R] - [(21, 53), (50, 82), (80, 112)]
p2s2 [R] - [(15, 47), (41, 73), (68, 100)]
p2s3 [L] - [(17, 49), (46, 78), (75, 107)]
p2s4 [L] - [(6, 38), (32, 64), (58, 90)]
p3s1 [L] - [(16, 48), (47, 79), (75, 107)]
p3s2 [R] - [(11, 43), (37, 69), (64, 96)]
p3s3 [L] - [(17, 49), (48, 80), (78, 110)]
p3s4 [R] - [(20, 52), (45, 77), (71, 103)]
p4s1 [L] - [(21, 53), (52, 84)]
p4s2 [L] - [(19, 51), (50, 82)]
p4s3 [R] - [(19, 51), (51, 83)]
p4s4 [R] - [(18, 50), (49, 81)]
p5s1 [R] - [(21, 53), (53, 85), (85, 117)]
p5s2 [R] - [(25, 57), (58, 90), (90, 122)]
p5s3 [R] - [(23, 55), (56, 88), (88, 120)]
p5s4 [R] - [(24, 56), (57, 89), (88, 120)]
p6s1 [L] - [(14, 46), (41, 73)]
p6s2 [L] - [(22, 54), (48, 80), (75, 107)]
p6s3 [R] - [(17, 49), (44, 76), (70, 102)]
p6s4 [L] - [(19, 51), (45, 77), (69, 101)]
p7s1 [R] - [(16, 48), (45, 77), (71, 103)]
p7s2 

In [5]:
for key in results.keys():
    if len(results[key]) != 32:
        print(key)

In [6]:
results['p1s1c0'][0]

{'lfemur': [8.636361122131348, 3.2603681087493896, -0.04507499933242798],
 'ltibia': [7.651093482971191, 1.7556756734848022, -0.2886623740196228],
 'lfoot': [7.5678863525390625, 0.20506787300109863, -0.26919037103652954],
 'ltoes': [7.064680099487305, 0.16790089011192322, -0.3366137146949768],
 'rfemur': [8.733498573303223, 3.2108194828033447, -1.0741034746170044],
 'rtibia': [9.425844192504883, 1.5164947509765625, -0.8321117758750916],
 'rfoot': [10.37553596496582, 0.258259654045105, -0.7889801859855652],
 'rtoes': [9.907983779907227, 0.11361396312713623, -0.7628710865974426],
 'lhumerus': [8.492287635803223, 4.93408203125, 0.05469483137130737],
 'lradius': [8.902358055114746, 3.810229778289795, 0.04463689401745796],
 'lwrist': [8.967071533203125, 3.182685375213623, 0.1379472017288208],
 'rhumerus': [8.376017570495605, 4.924999237060547, -1.0596261024475098],
 'rradius': [8.828798294067383, 3.7279603481292725, -1.208672046661377],
 'rwrist': [8.768996238708496, 3.1097776889801025, -1.

In [7]:
results['p21s4c1'][0]

{'lfemur': [-1.351020097732544, 3.6721510887145996, 0.1937260627746582],
 'ltibia': [-0.8942012786865234, 2.3452000617980957, 0.04692365229129791],
 'lfoot': [-0.6627256274223328, 0.3631434440612793, 0.14156977832317352],
 'ltoes': [-0.22914525866508484, 0.17698222398757935, -0.1823148876428604],
 'rfemur': [-0.88521409034729, 3.7109951972961426, 1.2990283966064453],
 'rtibia': [-1.3340295553207397, 2.0953052043914795, 1.310787320137024],
 'rfoot': [-2.3412094116210938, 0.8130124807357788, 1.6428189277648926],
 'rtoes': [-2.053978443145752, 0.2579759359359741, 1.507323980331421],
 'lhumerus': [-1.6414228677749634, 5.7468485832214355, -0.07997077703475952],
 'lradius': [-1.7031549215316772, 4.523014068603516, -0.019126255065202713],
 'lwrist': [-1.6469796895980835, 3.8142058849334717, -0.03867185860872269],
 'rhumerus': [-0.8580227494239807, 5.873493194580078, 1.3033502101898193],
 'rradius': [-0.9426620602607727, 4.658289909362793, 1.5708858966827393],
 'rwrist': [-0.7213826179504395, 

In [8]:
with open("./datasets/mocap/dataset_mocap_cycles_split.json", "w") as f:
    json.dump(results, f, indent=4)

In [9]:
import json
from utils.gait_parameters_extractor_v2 import GaitParametersExtractorV2
from utils.gait_parameters_extractor import CoordinatesIdx

def separate_gait_sequences_to_cycles(data_3d, 
                                      window_size: int = 32,
                                      minima_window_size:int = 10,
                                      coordinates_index: CoordinatesIdx = CoordinatesIdx(2, 0, 1), # not neccessary for this usage of this class
                                      scale_factor: int = 255, # not neccessary for this usage of this class
                                      print_stats: bool = True,
                                     ):
    results = {}
    cum_step_frames = []
    side = ''
    
    for seq_key in list(data_3d.keys()):
        gpe = GaitParametersExtractorV2(data_3d[seq_key], coordinates_index, scale_factor, minima_window_size)
        if len(gpe.l_steps)>1 and len(gpe.r_steps)>1:
            steps_sequence = []
            if gpe.l_steps[0] < gpe.r_steps[0]:
                cum_step_frames += _calc_step_frames(gpe.l_steps)
                steps_sequence = _fragment_step_frames(gpe.l_steps, window_size)
                side = 'L'
            else:
                cum_step_frames += _calc_step_frames(gpe.r_steps)
                steps_sequence = _fragment_step_frames(gpe.r_steps, window_size)
                side = 'R'

            if print_stats:
                print(f"{seq_key} [{side}] - {steps_sequence}")
            for i, (start_frame, end_frame) in enumerate(steps_sequence):
                results[f"{seq_key}c{i}"] = gpe.seq_params[start_frame:end_frame]

    if print_stats:
        print("   Steps: ", len(cum_step_frames))
        print("    Mean: ", sum(cum_step_frames)/len(cum_step_frames))
        print("     Max: ", max(cum_step_frames))
        print("     Min: ", min(cum_step_frames))
        print("  Median: ", sorted(cum_step_frames)[len(cum_step_frames)//2])
        print(" Over 32: ", sum(1 for step in cum_step_frames if step>32))

    return results

In [10]:
triang_data_file = "./datasets/yolo/dataset_v3.json"

with open(triang_data_file, 'r') as file:
    triangulation_data = json.load(file)

results = separate_gait_sequences_to_cycles(triangulation_data, window_size=32,minima_window_size=14, coordinates_index=CoordinatesIdx(0, 1, 2))

with open("./datasets/yolo/dataset_yolo_triang_cycles_split.json", "w") as f:
    json.dump(results, f, indent=4)

p1s1 [L] - [(27, 59), (58, 90)]
p1s2 [R] - [(16, 48), (44, 76)]
p1s3 [L] - [(22, 54), (53, 85)]
p1s4 [L] - [(25, 57), (54, 86)]
p2s1 [R] - [(28, 60), (58, 90), (85, 117)]
p2s2 [R] - [(18, 50), (45, 77), (73, 105)]
p2s3 [L] - [(22, 54), (52, 84), (78, 110)]
p2s4 [R] - [(23, 55), (50, 82), (76, 108)]
p3s1 [R] - [(13, 45), (39, 71), (64, 96)]
p3s2 [R] - [(12, 44), (39, 71)]
p3s3 [L] - [(28, 60), (53, 85), (87, 119)]
p3s4 [L] - [(34, 66), (60, 92)]
p4s1 [L] - [(27, 59), (60, 92)]
p4s2 [L] - [(24, 56), (54, 86)]
p4s3 [R] - [(28, 60), (61, 93)]
p4s4 [R] - [(24, 56), (55, 87)]
p5s1 [R] - [(22, 54), (55, 87), (87, 119)]
p5s2 [L] - [(16, 48), (44, 76), (77, 109), (108, 140)]
p5s3 [L] - [(16, 48), (45, 77), (73, 105)]
p5s4 [R] - [(32, 64), (59, 91), (91, 123)]
p6s1 [L] - [(23, 55), (45, 77)]
p6s2 [L] - [(23, 55), (50, 82), (76, 108)]
p6s3 [R] - [(20, 52), (48, 80), (72, 104)]
p6s4 [L] - [(24, 56), (46, 78)]
p7s1 [L] - [(12, 44), (40, 72), (65, 97)]
p7s2 [R] - [(21, 53), (49, 81), (74, 106)]
p7s3

In [11]:
triang_data_file = "./datasets/yolo/dataset_best_cameras.json"

with open(triang_data_file, 'r') as file:
    triangulation_data = json.load(file)

results = separate_gait_sequences_to_cycles(triangulation_data, window_size=32,minima_window_size=14, coordinates_index=CoordinatesIdx(0, 1, 2))

with open("./datasets/yolo/dataset_yolo_triang_best_cameras_cycles_split.json", "w") as f:
    json.dump(results, f, indent=4)

p1s1 [L] - [(29, 61), (55, 87)]
p1s2 [R] - [(12, 44), (43, 75)]
p1s3 [L] - [(24, 56), (57, 89), (84, 116)]
p1s4 [R] - [(14, 46), (42, 74)]
Falied to find proper step frame keys
p2s2 [R] - [(18, 50), (49, 81), (73, 105)]
p2s3 [L] - [(20, 52), (49, 81), (78, 110)]
p2s4 [L] - [(11, 43), (34, 66), (61, 93)]
p3s1 [R] - [(8, 40), (34, 66), (64, 96)]
p3s2 [R] - [(11, 43), (39, 71)]
p3s3 [L] - [(26, 58), (57, 89), (92, 124)]
p3s4 [R] - [(21, 53), (49, 81)]
p4s1 [L] - [(26, 58), (58, 90)]
p4s2 [L] - [(24, 56), (54, 86)]
p4s3 [R] - [(26, 58), (59, 91)]
p4s4 [R] - [(24, 56), (56, 88)]
p5s1 [R] - [(20, 52), (59, 91), (91, 123)]
p5s2 [L] - [(16, 48), (46, 78), (78, 110), (110, 142)]
p5s3 [L] - [(13, 45), (48, 80), (80, 112)]
p5s4 [L] - [(14, 46), (44, 76), (76, 108)]
p6s1 [L] - [(16, 48), (43, 75)]
p6s2 [L] - [(23, 55), (49, 81), (75, 107)]
p6s3 [R] - [(20, 52), (48, 80), (72, 104)]
First left step recognized as probably marked incorrectly and removed
p6s4 [L] - [(45, 77)]
p7s1 [L] - [(13, 45), (39